# 📊 Day 02a: ER Model — Entities, Relationships & ER Diagrams

---

## 🎯 What You'll Master Today
- Entities, attributes (simple, composite, multivalued, derived)
- Relationships & cardinality (1:1, 1:N, M:N)
- Weak entities & identifying relationships
- ER to relational mapping rules

---

```
╔══════════════════════════════════════════════════════════════╗
║                  ER DIAGRAM NOTATION                         ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  [Rectangle]     = Entity (Student, Course)                  ║
║  ((Double Rect))  = Weak Entity (Dependent)                  ║
║  (Oval)          = Attribute                                 ║
║  ((Double Oval))  = Multivalued Attribute (phone numbers)    ║
║  (Dashed Oval)    = Derived Attribute (age from DOB)         ║
║  <Diamond>       = Relationship                              ║
║  <<Double Dia>>   = Identifying Relationship (weak entity)   ║
║                                                              ║
║  Example:                                                    ║
║                                                              ║
║  [STUDENT] ——< enrolls >—— [COURSE]                         ║
║     |              M:N        |                              ║
║   (name)                    (title)                          ║
║   (roll_no) PK              (code) PK                       ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Attribute Types
# ============================================

attribute_types = [
    ("Simple",       "Atomic, indivisible",              "age, salary, roll_no"),
    ("Composite",    "Can be divided into sub-parts",    "name → (first, middle, last)\n" +
     "                                                      address → (street, city, zip)"),
    ("Single-valued","One value per entity",              "SSN, DOB"),
    ("Multivalued",  "Multiple values per entity",       "phone_numbers, email_addresses"),
    ("Derived",      "Computed from other attributes",   "age (from DOB), total_marks (from marks)"),
    ("Key",          "Uniquely identifies entity",       "roll_no, employee_id"),
]

print("  Attribute Types in ER Model")
print(f"  {'Type':<15} {'Description':<35} {'Examples'}")
print("  " + "─" * 80)
for atype, desc, examples in attribute_types:
    print(f"  {atype:<15} {desc:<35} {examples}")

In [ ]:
# ============================================
# 2. Cardinality & ER to Relational Mapping
# ============================================

cardinality_rules = [
    {
        "type": "1:1 (One-to-One)",
        "example": "Person ←→ Passport",
        "mapping": "Add FK in EITHER table (prefer the one with total participation)",
        "sql": """ALTER TABLE passport ADD person_id INT UNIQUE REFERENCES person(id);"""
    },
    {
        "type": "1:N (One-to-Many)",
        "example": "Department ←→ Employee (1 dept has many employees)",
        "mapping": "Add FK in the MANY side (employee gets dept_id)",
        "sql": """ALTER TABLE employee ADD dept_id INT REFERENCES department(id);"""
    },
    {
        "type": "M:N (Many-to-Many)",
        "example": "Student ←→ Course (students take many courses)",
        "mapping": "Create JUNCTION TABLE with both FKs as composite PK",
        "sql": """CREATE TABLE enrollment (
    student_id INT REFERENCES student(id),
    course_id  INT REFERENCES course(id),
    grade      CHAR(1),
    PRIMARY KEY (student_id, course_id)
);"""
    },
]

for card in cardinality_rules:
    print(f"  {'═' * 55}")
    print(f"  {card['type']}")
    print(f"  Example: {card['example']}")
    print(f"  Mapping Rule: {card['mapping']}")
    print(f"  SQL:")
    for line in card['sql'].strip().split('\n'):
        print(f"    {line}")
    print()

In [ ]:
# ============================================
# 3. Weak Entity Example
# ============================================

print("  Weak Entity — Depends on Strong Entity")
print("  " + "═" * 55)
print()
print("  A WEAK entity:")
print("    • Cannot be uniquely identified by its own attributes")
print("    • Depends on a STRONG (owner) entity")
print("    • Has a PARTIAL KEY (discriminator)")
print("    • Total participation in identifying relationship")
print()
print("  Example: Employee (strong) & Dependent (weak)")
print("  ─" * 25)
print("    Employee: emp_id (PK), name, salary")
print("    Dependent: dep_name (partial key), relationship, DOB")
print("    Dependent PK = (emp_id, dep_name) — needs owner's key!")
print()

# SQL for weak entity
weak_sql = """
  CREATE TABLE dependent (
      emp_id     INT REFERENCES employee(emp_id) ON DELETE CASCADE,
      dep_name   VARCHAR(50),
      relation   VARCHAR(20),
      dob        DATE,
      PRIMARY KEY (emp_id, dep_name)  -- composite PK!
  );
"""
print("  SQL:")
print(weak_sql)
print("  💡 ON DELETE CASCADE — if employee deleted, dependents go too")
print("  💡 Weak entity ALWAYS has total participation (double line in ER)")

In [ ]:
# ============================================
# 4. ER Design Practice — University DB
# ============================================

print("  ER Design Exercise: University Database")
print("  " + "═" * 55)
print()
print("  Entities & Attributes:")
print("    STUDENT:    roll_no(PK), name, DOB, *age(derived), {phone}(multivalued)")
print("    COURSE:     code(PK), title, credits")
print("    PROFESSOR:  prof_id(PK), name, department")
print("    DEPARTMENT: dept_id(PK), name, building")
print()
print("  Relationships:")
print("    STUDENT  ——M:N——  COURSE    (enrolls, with grade attribute)")
print("    PROFESSOR ——1:N—— COURSE    (teaches)")
print("    DEPARTMENT——1:N—— PROFESSOR (employs)")
print("    STUDENT  ——N:1——  DEPARTMENT(belongs_to)")
print()
print("  Resulting Tables:")
print("    student(roll_no, name, DOB, dept_id FK)")
print("    course(code, title, credits, prof_id FK)")
print("    professor(prof_id, name, dept_id FK)")
print("    department(dept_id, name, building)")
print("    enrollment(roll_no FK, code FK, grade)  ← junction for M:N")
print("    student_phone(roll_no FK, phone)         ← for multivalued")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is a weak entity? Give example | Entity that can't be identified by its own attributes. Dependent, order_item |
| 2 | How to map M:N relationship? | Create junction table with both FKs as composite PK |
| 3 | Total vs partial participation? | Total: every entity must participate. Partial: some may not |
| 4 | How to handle multivalued attributes? | Separate table with FK back to parent |
| 5 | Design ER for e-commerce | User, Product, Order, OrderItem(weak), Category, Review — classic interview Q |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Entity = noun, Attribute = property, Relationship = verb │
## │  • 1:1 → FK in either. 1:N → FK in many. M:N → junction │
## │  • Weak entity: partial key + owner's PK = composite PK  │
## │  • Multivalued attr → separate table                     │
## │  • Derived attr → don't store, compute on the fly         │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Keys & Constraints** — PK, FK, unique, CHECK, referential integrity